### Starting the silos 2018 dataset

In [1]:
import pandas as pd
import os 
import numpy as np
from sklearn.model_selection import train_test_split
import numpy as np 

In [23]:
data = pd.read_parquet('../datasets/cic-ids2018dataset/datasets/Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet')
data.head(5)

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,141385,9,7,553,3773.0,202,0,61.444443,87.534439,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1,6,281,2,1,38,0.0,38,0,19.000000,26.870058,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
2,6,279824,11,15,1086,10527.0,385,0,98.727272,129.392502,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
3,6,132,2,0,0,0.0,0,0,0.000000,0.000000,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
4,6,274016,9,13,1285,6141.0,517,0,142.777771,183.887726,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign


In [3]:
### Combining all the datasets 
silo_path = '../datasets/cic-ids2018dataset/datasets/'
datasets = []

for filename in os.listdir(silo_path):
    if filename.endswith('.parquet'):
        filepath = os.path.join(silo_path, filename) ### Creating a file path for each directory
        data = pd.read_parquet(filepath)
        datasets.append(data)
    
combined_datasets = pd.concat(datasets, ignore_index=True)
combined_datasets.head(5)
print(combined_datasets.shape[0])

6659532


In [4]:
print(combined_datasets['Label'].value_counts())

Label
Benign                      5329008
DDoS attacks-LOIC-HTTP       575364
DDOS attack-HOIC             198861
DoS attacks-Hulk             145199
Bot                          144535
Infilteration                118483
SSH-Bruteforce                94048
DoS attacks-GoldenEye         41406
DoS attacks-Slowloris          9908
DDOS attack-LOIC-UDP           1730
Brute Force -Web                568
Brute Force -XSS                229
SQL Injection                    85
DoS attacks-SlowHTTPTest         55
FTP-BruteForce                   53
Name: count, dtype: int64


In [5]:
## Only the attack samples 
attack_data = combined_datasets[combined_datasets['Label'] != 'Benign']
benign_data = combined_datasets[combined_datasets['Label'] == 'Benign']

In [6]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
NUM_CLIENTS = 4
ALPHA = 0.5
client_indices = [[] for _ in range(NUM_CLIENTS)]
for label in attack_data['Label'].unique():    
    indices = attack_data[attack_data['Label'] == label].index.to_numpy().copy() ## This returns the row number of each attack type sample
    np.random.shuffle(indices)
    proportions = np.random.dirichlet([ALPHA] * NUM_CLIENTS)
    split_points = (np.cumsum(proportions)[:-1] * len(indices)).astype(int)
    splits = np.split(indices, split_points)
    for client_id in range(NUM_CLIENTS):
        client_indices[client_id].extend(splits[client_id])

### each silo has sttack indices

In [7]:
client_datasets = []
for client_id in range(NUM_CLIENTS):
    attack_count = len(client_indices[client_id])
    benign_sample = benign_data.sample(n = attack_count, random_state=RANDOM_SEED)
    client_dataset = pd.concat(
    [
        attack_data.loc[client_indices[client_id]],
        benign_sample
    ])
    client_datasets.append(client_dataset)
    

In [8]:
for each in client_datasets:
    print(each.shape)

(1336088, 78)
(290252, 78)
(648204, 78)
(386504, 78)


In [9]:
def binary_silos_converter(df):
    df = df.copy()
    df['Label_Binary'] = df['Label'].apply(lambda x: 0 if x == 'Benign' else 1)
    df = df.drop(columns = 'Label')
    return df 

In [10]:
pd1 = client_datasets[0]
pd1_binary = binary_silos_converter(pd1)
print(pd1_binary['Label_Binary'].value_counts())
pd1_binary.to_csv('../silos_datasets/silos_datasets2018/SiloBinaryOne.csv')

Label_Binary
1    668044
0    668044
Name: count, dtype: int64


In [11]:
pd2 = client_datasets[2]
pd2_binary = binary_silos_converter(pd2)
print(pd2_binary['Label_Binary'].value_counts())
pd2_binary.to_csv('../silos_datasets/silos_datasets2018/SiloBinaryTwo.csv')

Label_Binary
1    324102
0    324102
Name: count, dtype: int64


In [12]:
pd3 = client_datasets[3]
pd3_binary = binary_silos_converter(pd3)
print(pd3_binary['Label_Binary'].value_counts())
pd3_binary.to_csv('../silos_datasets/silos_datasets2018/SiloBinaryThree.csv')

Label_Binary
1    193252
0    193252
Name: count, dtype: int64


In [13]:
pd4 = client_datasets[1]
pd4_binary = binary_silos_converter(pd4)
print(pd4_binary['Label_Binary'].value_counts())
pd4_binary.to_csv('../silos_datasets/silos_datasets2018/SiloBinaryFour.csv')

Label_Binary
1    145126
0    145126
Name: count, dtype: int64


In [14]:
### Binary combined datasets
binary_combined_dataset = pd.concat([pd1_binary, pd2_binary, pd3_binary, pd4_binary], ignore_index = True)
binary_combined_dataset.to_csv('../silos_datasets/silos_datasets2018/combined_dataset.csv')

In [15]:
### Creating the train test split for each silos

from sklearn.model_selection import train_test_split ## Imported the train test split library
from sklearn.preprocessing import StandardScaler
def train_validation_test(data, silo_name, random_state = RANDOM_SEED):
    scaler = StandardScaler()
    X = data.drop(columns=['Label_Binary'])
    y = data['Label_Binary']
    silo_path = '../silos_datasets/silos_datasets2018/'
    sending_path = '../FederatedAvg/cicids2018clientdata/nids/'
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3,random_state=random_state, stratify = y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, random_state=random_state, stratify=y_temp)

    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)
    X_train_scaled.to_csv(f'{sending_path}{silo_name}_X_train.csv', index=False)
    y_train.to_csv(f'{sending_path}{silo_name}_y_train.csv', index=False)
    X_test_scaled.to_csv(f'{sending_path}{silo_name}_X_test.csv', index=False)
    y_test.to_csv(f'{sending_path}{silo_name}_y_test.csv', index=False)
    X_val_scaled.to_csv(f'{sending_path}{silo_name}_X_val.csv', index=False)
    y_val.to_csv(f'{sending_path}{silo_name}_y_val.csv', index=False)

    return X_train_scaled, y_train, X_test_scaled, y_test, X_val_scaled, y_val

In [16]:
train_validation_test(pd1_binary,'siloOne')
train_validation_test(pd2_binary,'siloTwo')
train_validation_test(pd3_binary,'siloThree')
train_validation_test(pd4_binary,'siloFour')

(        Protocol  Flow Duration  Total Fwd Packets  Total Backward Packets  \
 0      -0.368391      -0.204839          -0.046341               -0.009583   
 1      -0.368391       2.663041          -0.046687               -0.029912   
 2      -0.368391      -0.381740          -0.046687               -0.029912   
 3      -0.368391      -0.381766          -0.046687               -0.029912   
 4      -0.368391      -0.378811          -0.046514               -0.029912   
 ...          ...            ...                ...                     ...   
 203171 -0.368391      -0.379758          -0.046687               -0.029912   
 203172  2.527328      -0.381791          -0.046860               -0.024829   
 203173 -0.368391      -0.381409          -0.046514               -0.009583   
 203174 -0.368391      -0.380461          -0.046687               -0.029912   
 203175 -0.368391      -0.222301          -0.046168               -0.014665   
 
         Fwd Packets Length Total  Bwd Packets Len

In [20]:
combined_datasets.head(5)

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,888751,11,11,1249,1969.0,736,0,113.545456,220.896072,...,32,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign
1,0,112642816,3,0,0,0.0,0,0,0.000000,0.000000,...,0,0.0,0.0,0.0,0.0,56300000.0,7.071068,56300000.0,56300000.0,Benign
2,0,112642712,3,0,0,0.0,0,0,0.000000,0.000000,...,0,0.0,0.0,0.0,0.0,56300000.0,18.384777,56300000.0,56300000.0,Benign
3,0,112642648,3,0,0,0.0,0,0,0.000000,0.000000,...,0,0.0,0.0,0.0,0.0,56300000.0,5.656854,56300000.0,56300000.0,Benign
4,0,112642702,3,0,0,0.0,0,0,0.000000,0.000000,...,0,0.0,0.0,0.0,0.0,56300000.0,65.053825,56300000.0,56300000.0,Benign
